In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv

In [14]:
load_dotenv("../.env",override=True)
data_path = os.getenv("Data_path_featured")

In [15]:
df=pd.read_csv(data_path)
df.head()

,CustomerID,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,InternetService,OnlineSecurity,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,DSL,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,DSL,Yes,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,DSL,Yes,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,DSL,Yes,Yes,One year,No,Bank transfer,42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,Fiber optic,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [16]:
x=df.drop(['Churn','CustomerID'],axis=1)
y=df["Churn"]

In [17]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2, random_state=42,stratify=y)

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler

In [19]:
x.columns

Index(['Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Tenure',
       'PhoneService', 'InternetService', 'OnlineSecurity', 'TechSupport',
       'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges'],
      dtype='object')

In [26]:
binary_colm = ['SeniorCitizen','Partner','Dependents','OnlineSecurity','TechSupport','PaperlessBilling']
multi_cat_colm = ['InternetService', 'PaymentMethod', 'Contract']
numerical_colm = ['Tenure', 'MonthlyCharges', 'TotalCharges']

In [27]:
binary_pipe = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))])

multi_cat_pipe = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))])

numerical_pipe = Pipeline(steps=[
    ('scaler', RobustScaler())])

In [28]:
preprocessor = ColumnTransformer(transformers=[
    ('bin', binary_pipe, binary_colm),
    ('multi', multi_cat_pipe, multi_cat_colm),
    ('num', numerical_pipe, numerical_colm)])

In [29]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

In [30]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())}

In [31]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    scores = cross_validate(estimator=pipe,X=x_train, y=y_train, cv=cv, scoring=['roc_auc', 'f1', 'recall', 'precision'])
    print(f"\n--- {name} ---")
    for k, v in scores.items():
        if 'test' in k:
            print(f"{k}: {v.mean():.3f} (+/- {v.std():.3f})")


--- Logistic Regression ---
test_roc_auc: 0.844 (+/- 0.012)
test_f1: 0.624 (+/- 0.018)
test_recall: 0.798 (+/- 0.030)
test_precision: 0.513 (+/- 0.016)

--- Random Forest ---
test_roc_auc: 0.819 (+/- 0.011)
test_f1: 0.546 (+/- 0.025)
test_recall: 0.478 (+/- 0.022)
test_precision: 0.638 (+/- 0.032)

--- XGBoost ---
test_roc_auc: 0.823 (+/- 0.010)
test_f1: 0.602 (+/- 0.018)
test_recall: 0.674 (+/- 0.025)
test_precision: 0.544 (+/- 0.015)


In [32]:
from sklearn.model_selection import RandomizedSearchCV

#Logistic Regression
lr_param_dist = {'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear'] }

lr_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))])

lr_search = RandomizedSearchCV(lr_pipe, lr_param_dist, n_iter=12, scoring='roc_auc',cv=cv, random_state=42, n_jobs=-1)
lr_search.fit(x_train, y_train)
print("LR best params:", lr_search.best_params_)
print("LR best CV ROC-AUC:", lr_search.best_score_)

#Random Forest 
rf_param_dist = {
    'classifier__n_estimators': [200, 300, 500, 700],
    'classifier__max_depth': [5, 10, 15, 20, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2']}

rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))])

rf_search = RandomizedSearchCV(rf_pipe, rf_param_dist, n_iter=30, scoring='roc_auc',cv=cv, random_state=42, n_jobs=-1)
rf_search.fit(x_train, y_train)
print("RF best params:", rf_search.best_params_)
print("RF best CV ROC-AUC:", rf_search.best_score_)

LR best params: {'classifier__solver': 'liblinear', 'classifier__penalty': 'l2', 'classifier__C': 100}
LR best CV ROC-AUC: 0.844274764591097
RF best params: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 4, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 5}
RF best CV ROC-AUC: 0.8452340590093943


In [33]:
param_dist = {
    'classifier__max_depth': [3,4,5,6,7],
    'classifier__learning_rate': [0.01,0.05,0.1,0.2],
    'classifier__n_estimators': [100,200,300,500],
    'classifier__min_child_weight': [1,3,5],
    'classifier__subsample': [0.7,0.8,0.9,1.0],
    'classifier__colsample_bytree': [0.7,0.8,0.9,1.0],}

xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, eval_metric='logloss',
                                  scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()))])

search = RandomizedSearchCV(xgb_pipe, param_dist, n_iter=50, scoring='roc_auc',cv=cv, random_state=42, n_jobs=-1)
search.fit(x_train, y_train)

print(search.best_params_)
print('Best CV ROC-AUC:', search.best_score_)


{'classifier__subsample': 0.9, 'classifier__n_estimators': 500, 'classifier__min_child_weight': 1, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.01, 'classifier__colsample_bytree': 0.7}
Best CV ROC-AUC: 0.8482682086538123


In [34]:
candidates = {
    'Logistic Regression': lr_search,
    'Random Forest': rf_search,
    'XGBoost': search}

for name, s in candidates.items():
    print(f"{name}: CV ROC-AUC = {s.best_score_:.4f}")

best_name = max(candidates, key=lambda k: candidates[k].best_score_)
best_pipe = candidates[best_name].best_estimator_
print(f"\nOverall best model: {best_name}")

Logistic Regression: CV ROC-AUC = 0.8443
Random Forest: CV ROC-AUC = 0.8452
XGBoost: CV ROC-AUC = 0.8483

Overall best model: XGBoost


In [35]:
from sklearn.metrics import precision_recall_curve, classification_report,roc_auc_score,confusion_matrix


y_proba = best_pipe.predict_proba(x_test)[:, 1]
prec, rec, thresh = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_t = thresh[f1_scores.argmax()]
print('Best threshold:', round(best_t, 3))

y_pred_tuned = (y_proba >= best_t).astype(int)
print(classification_report(y_test, y_pred_tuned))
print('ROC-AUC:', round(roc_auc_score(y_test, y_proba), 4))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_tuned))

Best threshold: 0.625
              precision    recall  f1-score   support

           0       0.88      0.83      0.85      1035
           1       0.59      0.69      0.64       374

    accuracy                           0.79      1409
   macro avg       0.73      0.76      0.74      1409
weighted avg       0.80      0.79      0.79      1409

ROC-AUC: 0.8454
Confusion Matrix:
 [[854 181]
 [115 259]]


In [36]:
import joblib

model_path = os.path.join(r'E:\coding\PROJECTS\customer_churn_prediction\models', 'churn_model_XGB.pkl')
joblib.dump(best_pipe, model_path)

['E:\\coding\\PROJECTS\\customer_churn_prediction\\models\\churn_model_XGB.pkl']